# Promedio de tiempos reales y atrasos hacia Estación Corregidora

Este notebook complementa `RutasQroBus.ipynb`. Parte de los horarios programados del GTFS local, consulta **Google Maps Routes API** para obtener la ETA en transporte público desde las paradas hacia la estación Corregidora, conserva observaciones históricas y calcula una corrección promedio.

El resultado principal es `data/tiempos_corregidos_google.csv`. `RutasQroBus.ipynb` lo consume de manera opcional para que el filtro de 30 minutos use el tiempo corregido.


## 1. Alcance y definición de las métricas

Se manejan tres medidas distintas:

1. **Tiempo GTFS programado**: suma de los tiempos entre paradas según `stop_times.txt`.
2. **ETA de Google Transit**: duración estimada al momento de la consulta; puede incluir espera, caminata y transbordos.
3. **Atraso estimado (proxy)**: `max(ETA Google - tiempo GTFS, 0)`. Se promedia por ruta y se suma al cálculo GTFS, como se solicitó.

> **Limitación importante:** Google Routes API no entrega un campo de “atraso del camión” ni permite seleccionar un modelo de tráfico cuando `travelMode="TRANSIT"`. La ETA puede incorporar información actualizada cuando el proveedor de transporte la comparte, pero el diferencial contra GTFS también puede contener espera, caminata o transbordos. Por ello se etiqueta como **proxy**, no como atraso operativo observado. Para atraso vehicular exacto se necesita el feed GTFS-Realtime `TripUpdates` de la agencia.

Documentación oficial consultada:

- [Rutas en transporte público](https://developers.google.com/maps/documentation/routes/transit-route)
- [Matriz de rutas en transporte público](https://developers.google.com/maps/documentation/routes/transit-rm)
- [Límites y facturación](https://developers.google.com/maps/documentation/routes/usage-and-billing)
- [Políticas y atribución de Routes API](https://developers.google.com/maps/documentation/routes/policies)


## 2. Configuración segura

La llave nunca se escribe en el notebook. Se carga desde el archivo `.env` ubicado en la raíz del repositorio. Copia `.env.example` como `.env` y completa:

```dotenv
GOOGLE_MAPS_API_KEY=tu_llave
EJECUTAR_GOOGLE_MAPS=true
UMBRAL_ANALISIS_MIN=30
```

El cargador no sobrescribe variables que ya existan en el sistema. `.env` está excluido por `.gitignore`; solo `.env.example`, que no contiene secretos, puede versionarse. La consulta es facturable y permanece apagada salvo que `EJECUTAR_GOOGLE_MAPS=true`.


In [1]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import heapq
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import requests
from IPython.display import display

# Rutas portables: funciona desde scripts/ y desde la raíz del repositorio.
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
if not DATA_DIR.exists():
    raise FileNotFoundError("No se encontró la carpeta local data/.")

PROJECT_ROOT = DATA_DIR.parent.resolve()
ENV_PATH = PROJECT_ROOT / ".env"

def cargar_env(env_path):
    """Carga pares CLAVE=VALOR sencillos sin imprimir secretos."""
    if not env_path.exists():
        return False

    for numero_linea, linea_original in enumerate(
        env_path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        linea = linea_original.strip()
        if not linea or linea.startswith("#"):
            continue
        if linea.startswith("export "):
            linea = linea[7:].strip()
        if "=" not in linea:
            raise ValueError(f"Línea inválida en .env: {numero_linea}")

        clave, valor = linea.split("=", 1)
        clave, valor = clave.strip(), valor.strip()
        if len(valor) >= 2 and valor[0] == valor[-1] and valor[0] in {"'", '"'}:
            valor = valor[1:-1]
        os.environ.setdefault(clave, valor)

    return True

ENV_CARGADO = cargar_env(ENV_PATH)

CORREGIDORA_LAT = 20.600611
CORREGIDORA_LON = -100.402184

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "").strip() or None
EJECUTAR_API = os.getenv("EJECUTAR_GOOGLE_MAPS", "false").strip().lower() == "true"
UMBRAL_ANALISIS_MIN = float(os.getenv("UMBRAL_ANALISIS_MIN", "30"))
TAMANO_LOTE = 100
ESPERA_ENTRE_LOTES_SEG = 0.25
RETENCION_DIAS = 30

# None consulta todos los orígenes alcanzables. Use un entero para una prueba controlada.
MAX_PARADAS = None

if EJECUTAR_API and not API_KEY:
    raise RuntimeError(
        "EJECUTAR_GOOGLE_MAPS=true, pero falta GOOGLE_MAPS_API_KEY."
    )

print(f"Archivo .env cargado: {ENV_CARGADO} ({ENV_PATH})")
print(f"API key configurada: {bool(API_KEY)}")
print(f"Directorio de datos: {DATA_DIR.resolve()}")
print(f"Consulta a Google habilitada: {EJECUTAR_API}")


Archivo .env cargado: True (/Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/.env)
API key configurada: True
Directorio de datos: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data
Consulta a Google habilitada: True


## 3. Lectura y validación de los archivos GTFS

Se leen los cuatro archivos desde `data/` y se validan las columnas mínimas antes de calcular. Esta validación falla temprano si cambia el esquema de la fuente.


In [2]:
trips = pd.read_csv(DATA_DIR / "trips.txt")
stop_times = pd.read_csv(DATA_DIR / "stop_times.txt")
stops = pd.read_csv(DATA_DIR / "stops.txt")
routes = pd.read_csv(DATA_DIR / "routes.txt")

columnas_requeridas = {
    "trips": {"trip_id", "route_id"},
    "stop_times": {
        "trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"
    },
    "stops": {"stop_id", "stop_name", "stop_lat", "stop_lon"},
    "routes": {"route_id", "route_short_name", "route_long_name"},
}
tablas = {
    "trips": trips,
    "stop_times": stop_times,
    "stops": stops,
    "routes": routes,
}

for nombre, requeridas in columnas_requeridas.items():
    faltantes = requeridas - set(tablas[nombre].columns)
    if faltantes:
        raise ValueError(f"{nombre} no contiene las columnas: {sorted(faltantes)}")

if stops["stop_id"].duplicated().any():
    warnings.warn("Hay stop_id duplicados; se conservará el primer registro.")
    stops = stops.drop_duplicates("stop_id", keep="first").copy()

print(
    f"{len(stops):,} paradas, {len(routes):,} rutas, "
    f"{len(trips):,} viajes y {len(stop_times):,} horarios cargados."
)


2,694 paradas, 217 rutas, 32,536 viajes y 1,326,009 horarios cargados.


## 4. Tiempo programado hacia Corregidora

Los horarios GTFS admiten horas mayores a 24:00. Primero se convierten a segundos sin perder el cambio de día. Después se construye un grafo dirigido de segmentos consecutivos y se aplica Dijkstra sobre el grafo inverso para encontrar el menor tiempo programado desde cada parada hacia Corregidora.

Para evitar que una observación extrema gobierne un segmento, el peso de cada combinación `origen-destino-ruta` es su **mediana** programada.


In [ ]:
def gtfs_time_to_seconds(series):
    partes = series.astype(str).str.extract(
        r"^(?P<h>\d+):(?P<m>[0-5]\d):(?P<s>[0-5]\d)$"
    )
    if partes.isna().any(axis=None):
        ejemplos = series[partes.isna().any(axis=1)].head().tolist()
        raise ValueError(f"Horarios GTFS inválidos. Ejemplos: {ejemplos}")
    partes = partes.astype(int)
    return partes["h"] * 3600 + partes["m"] * 60 + partes["s"]


stop_times_work = stop_times.copy()
stop_times_work["arrival_sec"] = gtfs_time_to_seconds(
    stop_times_work["arrival_time"]
)
stop_times_work["departure_sec"] = gtfs_time_to_seconds(
    stop_times_work["departure_time"]
)
stop_times_work = stop_times_work.merge(
    trips[["trip_id", "route_id"]], on="trip_id", how="left", validate="many_to_one"
)
stop_times_work = stop_times_work.sort_values(
    ["trip_id", "stop_sequence"], kind="stable"
)

stop_times_work["next_stop_id"] = (
    stop_times_work.groupby("trip_id", sort=False)["stop_id"].shift(-1)
)
stop_times_work["next_arrival_sec"] = (
    stop_times_work.groupby("trip_id", sort=False)["arrival_sec"].shift(-1)
)

segmentos = stop_times_work.dropna(
    subset=["next_stop_id", "next_arrival_sec", "route_id"]
).copy()

segmentos["next_stop_id"] = segmentos["next_stop_id"].astype(stops["stop_id"].dtype)

segmentos["tiempo_segmento_min"] = (
    segmentos["next_arrival_sec"] - segmentos["departure_sec"]
) / 60

segmentos = segmentos[
    segmentos["tiempo_segmento_min"].between(0, 180, inclusive="both")
].copy()

segmentos_medianos = (
    segmentos.groupby(
        ["stop_id", "next_stop_id", "route_id"], as_index=False
    )["tiempo_segmento_min"]
    .median()
)

coords = stops[["stop_lat", "stop_lon"]].to_numpy()
distancia_cuadrada = (
    (coords[:, 0] - CORREGIDORA_LAT) ** 2
    + (coords[:, 1] - CORREGIDORA_LON) ** 2
)
corregidora_idx = int(np.argmin(distancia_cuadrada))
corregidora_stop_id = stops.iloc[corregidora_idx]["stop_id"]
corregidora_stop_name = stops.iloc[corregidora_idx]["stop_name"]

print(
    "Parada GTFS más cercana a Corregidora:",
    corregidora_stop_id,
    "-",
    corregidora_stop_name,
)


Parada GTFS más cercana a Corregidora: 3025 - Estío/Calle Dr. Manuel Domínguez


In [4]:
adj_reverse = defaultdict(list)
for fila in segmentos_medianos.itertuples(index=False):
    adj_reverse[fila.next_stop_id].append(
        (fila.stop_id, float(fila.tiempo_segmento_min), fila.route_id)
    )

distancias = {corregidora_stop_id: 0.0}
siguiente_parada = {}
ruta_del_salto = {}
heap = [(0.0, corregidora_stop_id)]

while heap:
    tiempo_actual, parada_actual = heapq.heappop(heap)
    if tiempo_actual > distancias.get(parada_actual, np.inf):
        continue

    for anterior, tiempo_segmento, route_id in adj_reverse.get(parada_actual, []):
        candidato = tiempo_actual + tiempo_segmento
        if candidato < distancias.get(anterior, np.inf):
            distancias[anterior] = candidato
            siguiente_parada[anterior] = parada_actual
            ruta_del_salto[anterior] = route_id
            heapq.heappush(heap, (candidato, anterior))


def reconstruir_camino(stop_id):
    camino = [stop_id]
    rutas_camino = []
    visitadas = {stop_id}
    actual = stop_id

    while actual != corregidora_stop_id and actual in siguiente_parada:
        rutas_camino.append(ruta_del_salto[actual])
        actual = siguiente_parada[actual]
        if actual in visitadas:
            raise RuntimeError(f"Ciclo inesperado al reconstruir {stop_id}.")
        visitadas.add(actual)
        camino.append(actual)

    return camino, rutas_camino


route_name_map = routes.set_index("route_id")["route_short_name"].to_dict()
registros_base = []

for stop_id, tiempo_gtfs in distancias.items():
    camino, rutas_camino = reconstruir_camino(stop_id)
    ruta_principal = rutas_camino[0] if rutas_camino else np.nan
    rutas_unicas = list(dict.fromkeys(rutas_camino))
    registros_base.append(
        {
            "stop_id": stop_id,
            "tiempo_gtfs_min": tiempo_gtfs,
            "route_id_principal": ruta_principal,
            "route_short_name": route_name_map.get(ruta_principal),
            "num_paradas_camino": len(camino),
            "num_rutas_camino": len(rutas_unicas),
            "camino_stop_ids": "|".join(map(str, camino)),
            "rutas_camino": "|".join(map(str, rutas_unicas)),
        }
    )

df_base = (
    pd.DataFrame(registros_base)
    .merge(
        stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]],
        on="stop_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("tiempo_gtfs_min")
    .reset_index(drop=True)
)

print(f"{len(df_base):,} paradas tienen camino programado hacia Corregidora.")
display(df_base.head(10))


2,259 paradas tienen camino programado hacia Corregidora.


,stop_id,tiempo_gtfs_min,route_id_principal,route_short_name,num_paradas_camino,num_rutas_camino,camino_stop_ids,rutas_camino,stop_name,stop_lat,stop_lon
0,3025,0.000000,NaN,None,1,0,3025,,Estío/Calle Dr. Manuel Domínguez,20.599573,-100.400492
1,2341,1.533333,344.0,C54,2,1,2341|3025,344,Av. Felipe Ángeles/Av. San Roque,20.603829,-100.399494
2,3295,2.216667,344.0,C54,3,1,3295|2341|3025,344,Felipe Ángeles/Fraternidad,20.605753,-100.399474
3,2340,2.733333,344.0,C54,4,1,2340|3295|2341|3025,344,Av. Felipe Ángeles/Plan de Ayala Poniente,20.607204,-100.399456
4,3294,3.583333,344.0,C54,5,1,3294|2340|3295|2341|3025,344,Av. Felipe Ángeles/Calle del Porvenir,20.609586,-100.399427
5,2339,4.200000,344.0,C54,6,1,2339|3294|2340|3295|2341|3025,344,Av. Felipe Ángeles/Felipe Ángeles 225,20.611300,-100.399656
6,2338,4.700000,344.0,C54,7,1,2338|2339|3294|2340|3295|2341|3025,344,Av. Felipe Ángeles/Estadística,20.612684,-100.399821
7,4256,5.366667,344.0,C54,8,1,4256|2338|2339|3294|2340|3295|2341|3025,344,Epigmenio González/Departamental Parques,20.612535,-100.401796
8,2337,7.000000,375.0,T07,8,2,2337|2338|2339|3294|2340|3295|2341|3025,375|344,Prol. Ezequiel Montes/Soriana Hiper,20.615980,-100.403161
9,3284,7.650000,324.0,C33,9,2,3284|4256|2338|2339|3294|2340|3295|2341|3025,324|344,Epigmenio González/Prol. Porvenir,20.607994,-100.406591


## 5. Consulta de Google Maps Routes API

Se utiliza `computeRouteMatrix` con una sola estación destino y hasta 100 paradas origen por lote. Antes de consultar se conservan solamente las paradas cuyo tiempo GTFS es menor o igual a `UMBRAL_ANALISIS_MIN` (30 minutos por defecto). Como el atraso aplicado nunca es negativo, una parada que ya supera ese umbral no puede entrar posteriormente en el resultado de 30 minutos.

El modo `TRANSIT` se restringe preferentemente a autobús. La respuesta solo solicita los campos necesarios para controlar costo, tamaño y trazabilidad.

Cada ejecución genera una instantánea con sello UTC. Las observaciones se acumulan temporalmente en `data/google_maps_transit_observaciones.csv`; así el promedio mejora al ejecutar el notebook en distintos días y horarios. El notebook elimina automáticamente filas mayores a `RETENCION_DIAS`. Esta retención es una salvaguarda técnica, no sustituye la revisión del acuerdo vigente de Google Maps aplicable al proyecto.

La función incluye reintentos exponenciales para errores temporales (`429` y `5xx`). Los errores permanentes se conservan en la salida en vez de ocultarse.


In [5]:
ROUTE_MATRIX_URL = (
    "https://routes.googleapis.com/distanceMatrix/v2:computeRouteMatrix"
)
FIELD_MASK = (
    "originIndex,destinationIndex,duration,distanceMeters,status,condition"
)


def duracion_google_a_minutos(valor):
    if valor is None or not str(valor).endswith("s"):
        return np.nan
    return float(str(valor)[:-1]) / 60


def crear_waypoint(latitud, longitud):
    return {
        "waypoint": {
            "location": {
                "latLng": {
                    "latitude": float(latitud),
                    "longitude": float(longitud),
                }
            }
        }
    }


def consultar_lote_google(df_lote, departure_time, max_intentos=4):
    payload = {
        "origins": [
            crear_waypoint(f.stop_lat, f.stop_lon)
            for f in df_lote.itertuples(index=False)
        ],
        "destinations": [
            crear_waypoint(CORREGIDORA_LAT, CORREGIDORA_LON)
        ],
        "travelMode": "TRANSIT",
        "departureTime": departure_time,
        "transitPreferences": {"allowedTravelModes": ["BUS"]},
    }
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": FIELD_MASK,
    }

    for intento in range(max_intentos):
        respuesta = requests.post(
            ROUTE_MATRIX_URL,
            headers=headers,
            json=payload,
            timeout=(10, 90),
        )
        if respuesta.status_code not in {429, 500, 502, 503, 504}:
            break
        if intento == max_intentos - 1:
            break
        time.sleep(2 ** intento)

    if not respuesta.ok:
        detalle = respuesta.text[:1_000]
        raise RuntimeError(
            f"Google Routes API respondió {respuesta.status_code}: {detalle}"
        )

    try:
        elementos = respuesta.json()
    except requests.JSONDecodeError as exc:
        raise RuntimeError("La respuesta de Google no es JSON válido.") from exc

    if not isinstance(elementos, list):
        raise RuntimeError(f"Formato inesperado de Google: {elementos}")

    ahora_utc = datetime.now(timezone.utc).isoformat()
    resultados = []

    for elemento in elementos:
        indice = elemento.get("originIndex")
        if indice is None or indice >= len(df_lote):
            continue
        origen = df_lote.iloc[int(indice)]
        status = elemento.get("status") or {}
        resultados.append(
            {
                "observed_at_utc": ahora_utc,
                "departure_time_utc": departure_time,
                "stop_id": origen["stop_id"],
                "route_id_principal": origen["route_id_principal"],
                "tiempo_gtfs_min": origen["tiempo_gtfs_min"],
                "google_eta_min": duracion_google_a_minutos(
                    elemento.get("duration")
                ),
                "google_distance_m": elemento.get("distanceMeters"),
                "condition": elemento.get("condition"),
                "status_code": status.get("code", 0),
                "status_message": status.get("message"),
            }
        )

    return pd.DataFrame(resultados)


In [6]:
df_consulta = df_base[
    df_base["stop_id"].ne(corregidora_stop_id)
    & df_base["tiempo_gtfs_min"].le(UMBRAL_ANALISIS_MIN)
].copy()
if MAX_PARADAS is not None:
    df_consulta = df_consulta.head(int(MAX_PARADAS)).copy()

print(
    f"Filtro GTFS aplicado: <= {UMBRAL_ANALISIS_MIN:g} minutos. "
    f"Paradas preparadas: {len(df_consulta):,}. "
    f"Elementos facturables estimados para esta ejecución: {len(df_consulta):,}."
)

observaciones_nuevas = []
departure_time = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

if EJECUTAR_API:
    total_lotes = int(np.ceil(len(df_consulta) / TAMANO_LOTE))
    for numero, inicio in enumerate(
        range(0, len(df_consulta), TAMANO_LOTE), start=1
    ):
        lote = df_consulta.iloc[inicio : inicio + TAMANO_LOTE].reset_index(
            drop=True
        )
        resultado_lote = consultar_lote_google(lote, departure_time)
        observaciones_nuevas.append(resultado_lote)
        print(
            f"Lote {numero}/{total_lotes}: "
            f"{len(resultado_lote)} respuestas recibidas."
        )
        if numero < total_lotes:
            time.sleep(ESPERA_ENTRE_LOTES_SEG)
else:
    print(
        "Consulta omitida. Configure GOOGLE_MAPS_API_KEY y "
        "EJECUTAR_GOOGLE_MAPS=true para obtener una instantánea."
    )

df_nuevas = (
    pd.concat(observaciones_nuevas, ignore_index=True)
    if observaciones_nuevas
    else pd.DataFrame()
)


Filtro GTFS aplicado: <= 30 minutos. Paradas preparadas: 1,043. Elementos facturables estimados para esta ejecución: 1,043.


Lote 1/11: 100 respuestas recibidas.


Lote 2/11: 100 respuestas recibidas.


Lote 3/11: 100 respuestas recibidas.


Lote 4/11: 100 respuestas recibidas.


...

Lote 7/11: 100 respuestas recibidas.


Lote 8/11: 100 respuestas recibidas.


Lote 9/11: 100 respuestas recibidas.


Lote 10/11: 100 respuestas recibidas.


Lote 11/11: 43 respuestas recibidas.


## 6. Persistencia y control de calidad

Solo se consideran válidos los elementos con `condition="ROUTE_EXISTS"`, sin código de error y con duración numérica. El histórico conserva también los fallos para poder auditar cobertura y disponibilidad.


In [7]:
OBSERVACIONES_PATH = DATA_DIR / "google_maps_transit_observaciones.csv"

if OBSERVACIONES_PATH.exists():
    historico_previo = pd.read_csv(OBSERVACIONES_PATH)
    if "observed_at_utc" in historico_previo.columns:
        fechas = pd.to_datetime(
            historico_previo["observed_at_utc"], errors="coerce", utc=True
        )
        limite = pd.Timestamp.now(tz="UTC") - pd.Timedelta(
            days=RETENCION_DIAS
        )
        historico_previo = historico_previo[fechas.ge(limite)].copy()
else:
    historico_previo = pd.DataFrame()

if not df_nuevas.empty:
    df_observaciones = pd.concat(
        [historico_previo, df_nuevas], ignore_index=True
    )
    df_observaciones = df_observaciones.drop_duplicates(
        subset=["observed_at_utc", "stop_id"], keep="last"
    )
    df_observaciones.to_csv(OBSERVACIONES_PATH, index=False)
    print(f"Histórico actualizado: {OBSERVACIONES_PATH}")
else:
    df_observaciones = historico_previo.copy()

if df_observaciones.empty:
    df_validas = pd.DataFrame()
    print("Todavía no existen observaciones de Google para promediar.")
else:
    columnas_numericas = [
        "tiempo_gtfs_min",
        "google_eta_min",
        "status_code",
    ]
    for columna in columnas_numericas:
        df_observaciones[columna] = pd.to_numeric(
            df_observaciones[columna], errors="coerce"
        )

    df_validas = df_observaciones[
        df_observaciones["condition"].eq("ROUTE_EXISTS")
        & df_observaciones["status_code"].fillna(0).eq(0)
        & df_observaciones["google_eta_min"].notna()
        & df_observaciones["tiempo_gtfs_min"].notna()
    ].copy()

    df_validas["desviacion_eta_min"] = (
        df_validas["google_eta_min"] - df_validas["tiempo_gtfs_min"]
    )
    df_validas["atraso_estimado_min"] = (
        df_validas["desviacion_eta_min"].clip(lower=0)
    )

    cobertura = df_validas["stop_id"].nunique() / max(len(df_consulta), 1)
    print(f"Observaciones totales: {len(df_observaciones):,}")
    print(f"Observaciones válidas: {len(df_validas):,}")
    print(f"Cobertura de paradas de esta población: {cobertura:.1%}")
    print(f"Datos de rutas: Google Maps ©{datetime.now().year} Google")
    display(
        df_validas[
            [
                "observed_at_utc",
                "stop_id",
                "route_id_principal",
                "tiempo_gtfs_min",
                "google_eta_min",
                "desviacion_eta_min",
                "atraso_estimado_min",
            ]
        ].head(10)
    )


Histórico actualizado: ../data/google_maps_transit_observaciones.csv
Observaciones totales: 7,301
Observaciones válidas: 7,301
Cobertura de paradas de esta población: 100.0%
Datos de rutas: Google Maps ©2026 Google


/var/folders/06/082vc2px7v791h689yrjpmv80000gn/T/ipykernel_5539/2400048268.py:9: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  limite = pd.Timestamp.now(tz="UTC") - pd.Timedelta(


,observed_at_utc,stop_id,route_id_principal,tiempo_gtfs_min,google_eta_min,desviacion_eta_min,atraso_estimado_min
0,2026-09-14T16:47:07.157796+00:00,2982,255.0,14.650000,17.733333,3.083333,3.083333
1,2026-09-14T16:47:07.157796+00:00,2981,355.0,15.083333,14.500000,-0.583333,0.000000
2,2026-09-14T16:47:07.157796+00:00,3297,97.0,15.583333,16.600000,1.016667,1.016667
3,2026-09-14T16:47:07.157796+00:00,3222,323.0,14.766667,14.016667,-0.750000,0.000000
4,2026-09-14T16:47:07.157796+00:00,3294,344.0,3.583333,13.283333,9.700000,9.700000
5,2026-09-14T16:47:07.157796+00:00,3284,324.0,7.650000,21.000000,13.350000,13.350000
6,2026-09-14T16:47:07.157796+00:00,3571,336.0,14.500000,14.466667,-0.033333,0.000000
7,2026-09-14T16:47:07.157796+00:00,1920,325.0,16.000000,19.500000,3.500000,3.500000
8,2026-09-14T16:47:07.157796+00:00,3295,344.0,2.216667,5.466667,3.250000,3.250000
9,2026-09-14T16:47:07.157796+00:00,5196,326.0,13.783333,23.533333,9.750000,9.750000


## 7. Promedio del atraso y tiempo corregido

El atraso proxy se agrega por `route_id_principal`. La corrección solicitada es:

```text
tiempo_corregido = tiempo_GTFS + atraso_promedio_de_la_ruta
```

También se calcula la ETA media observada por parada como control. Cuando una ruta aún no tiene observaciones válidas no se inventa un atraso: queda en cero y `correccion_disponible=False`.


In [8]:
PROMEDIO_RUTAS_PATH = DATA_DIR / "promedio_atrasos_google_por_ruta.csv"
CORRECCIONES_PATH = DATA_DIR / "tiempos_corregidos_google.csv"

if df_validas.empty:
    promedio_por_ruta = pd.DataFrame(
        columns=[
            "route_id_principal",
            "observaciones",
            "paradas_observadas",
            "atraso_promedio_ruta_min",
            "desviacion_promedio_ruta_min",
        ]
    )
    correcciones = df_base.copy()
    correcciones["atraso_promedio_ruta_min"] = 0.0
    correcciones["google_eta_promedio_stop_min"] = np.nan
    correcciones["observaciones_stop"] = 0
    correcciones["correccion_disponible"] = False
    correcciones["tiempo_corregido_min"] = correcciones["tiempo_gtfs_min"]
else:
    promedio_por_ruta = (
        df_validas.dropna(subset=["route_id_principal"])
        .groupby("route_id_principal", as_index=False)
        .agg(
            observaciones=("stop_id", "size"),
            paradas_observadas=("stop_id", "nunique"),
            atraso_promedio_ruta_min=("atraso_estimado_min", "mean"),
            desviacion_promedio_ruta_min=("desviacion_eta_min", "mean"),
        )
    )
    promedio_por_ruta["route_short_name"] = promedio_por_ruta[
        "route_id_principal"
    ].map(route_name_map)

    promedio_por_stop = (
        df_validas.groupby("stop_id", as_index=False)
        .agg(
            google_eta_promedio_stop_min=("google_eta_min", "mean"),
            observaciones_stop=("google_eta_min", "size"),
        )
    )

    correcciones = (
        df_base.merge(
            promedio_por_ruta[
                ["route_id_principal", "atraso_promedio_ruta_min"]
            ],
            on="route_id_principal",
            how="left",
        )
        .merge(promedio_por_stop, on="stop_id", how="left")
    )
    correcciones["correccion_disponible"] = correcciones[
        "atraso_promedio_ruta_min"
    ].notna()
    correcciones["atraso_promedio_ruta_min"] = correcciones[
        "atraso_promedio_ruta_min"
    ].fillna(0)
    correcciones["observaciones_stop"] = correcciones[
        "observaciones_stop"
    ].fillna(0).astype(int)
    correcciones["tiempo_corregido_min"] = (
        correcciones["tiempo_gtfs_min"]
        + correcciones["atraso_promedio_ruta_min"]
    )

    promedio_por_ruta.to_csv(PROMEDIO_RUTAS_PATH, index=False)
    correcciones.to_csv(CORRECCIONES_PATH, index=False)
    print(f"Promedios exportados: {PROMEDIO_RUTAS_PATH}")
    print(f"Correcciones exportadas: {CORRECCIONES_PATH}")

display(promedio_por_ruta.sort_values("atraso_promedio_ruta_min", ascending=False).head(15))


Promedios exportados: ../data/promedio_atrasos_google_por_ruta.csv
Correcciones exportadas: ../data/tiempos_corregidos_google.csv


,route_id_principal,observaciones,paradas_observadas,atraso_promedio_ruta_min,desviacion_promedio_ruta_min,route_short_name
4,94.0,21,3,31.653571,31.653571,L55
90,408.0,42,6,30.109921,30.109921,C73
5,95.0,7,1,29.200000,29.200000,L56
70,361.0,21,3,28.730159,28.730159,L156
29,316.0,35,5,28.520476,28.520476,C71
27,283.0,63,9,26.925661,26.925661,C67
3,93.0,7,1,26.501190,26.501190,L54
77,372.0,7,1,26.492857,26.492857,T04
2,92.0,7,1,26.313095,26.313095,L53
11,128.0,28,4,25.417560,25.417560,L100


## 8. Paradas y rutas dentro de 30 minutos

Este reporte usa `tiempo_corregido_min`, no el horario GTFS sin ajustar. Se excluyen del resultado final las filas sin corrección disponible para evitar afirmar que están dentro de 30 minutos sin evidencia de Google. Para diagnóstico se muestra además cuántas quedaron pendientes de observación.


In [9]:
rutas_30_min = correcciones[
    correcciones["correccion_disponible"]
    & correcciones["tiempo_corregido_min"].le(30)
].copy()

rutas_30_min = rutas_30_min.sort_values(
    ["tiempo_corregido_min", "route_short_name", "stop_name"]
)

pendientes_30_gtfs = correcciones[
    ~correcciones["correccion_disponible"]
    & correcciones["tiempo_gtfs_min"].le(30)
]

print(f"Paradas confirmadas a 30 min o menos: {len(rutas_30_min):,}")
print(
    "Paradas que GTFS coloca a 30 min o menos pero aún no tienen corrección:",
    f"{len(pendientes_30_gtfs):,}",
)

columnas_reporte = [
    "stop_id",
    "stop_name",
    "route_id_principal",
    "route_short_name",
    "tiempo_gtfs_min",
    "atraso_promedio_ruta_min",
    "tiempo_corregido_min",
    "google_eta_promedio_stop_min",
    "observaciones_stop",
]
display(rutas_30_min[columnas_reporte].reset_index(drop=True))


Paradas confirmadas a 30 min o menos: 245
Paradas que GTFS coloca a 30 min o menos pero aún no tienen corrección: 1


,stop_id,stop_name,route_id_principal,route_short_name,tiempo_gtfs_min,atraso_promedio_ruta_min,tiempo_corregido_min,google_eta_promedio_stop_min,observaciones_stop
0,2341,Av. Felipe Ángeles/Av. San Roque,344.0,C54,1.533333,5.430403,6.963736,6.785714,7
1,3295,Felipe Ángeles/Fraternidad,344.0,C54,2.216667,5.430403,7.647070,9.409524,7
2,2340,Av. Felipe Ángeles/Plan de Ayala Poniente,344.0,C54,2.733333,5.430403,8.163736,7.800000,7
3,3294,Av. Felipe Ángeles/Calle del Porvenir,344.0,C54,3.583333,5.430403,9.013736,9.966667,7
4,2339,Av. Felipe Ángeles/Felipe Ángeles 225,344.0,C54,4.200000,5.430403,9.630403,9.935714,7
...,...,...,...,...,...,...,...,...,...
240,2332,Amanecer,350.0,C62,13.866667,15.958473,29.825140,30.335714,7
241,3545,Av. Peñuelas/Av. Paseo de la Constitución,336.0,C46,17.216667,12.644407,29.861074,33.850000,7
242,1439,Alameda,90.0,L51,17.700000,12.180138,29.880138,23.800000,7
243,3293,Av. Felipe Ángeles/3a. Felipe Ángeles,345.0,C55,23.450000,6.432338,29.882338,9.966667,7


## 9. Recomendación de operación

Para que el promedio represente horas pico y valle, ejecute este notebook de forma programada varias veces al día y conserve el histórico. Como mínimo conviene muestrear mañana, mediodía y tarde durante días hábiles y fines de semana.

Antes de automatizar a gran escala:

- configure una cuota diaria y alertas de presupuesto en Google Cloud;
- revise la cobertura de QroBus en Google Transit;
- no publique la llave ni el archivo de observaciones sin revisar las condiciones de uso de Google Maps;
- si se obtiene un feed GTFS-Realtime oficial, reemplace el proxy por el atraso de `TripUpdates`.
